# RAG LLM LawBot

## Install relevant packages

In [1]:
%%capture

!pip install unsloth
!pip install bitsandbytes
!pip install unsloth_zoo
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install -U sentence-transformers
!pip install -q langchain==0.1.20
!pip install -q langchain-community==0.0.38
!pip install -q chromadb==0.4.24
!pip install -q gradio==4.36.1
!pip install -q pymupdf==1.23.8
!pip install -q sentence-transformers==2.2.2
!pip install youtube_dl
!pip install whisper
!pip install -U huggingface_hub
!pip install -U sentence-transformers
!pip install rank-bm25
!pip install rouge-score
!pip install bert-score
!pip install sentence-transformers
!pip install Sastrawi

## Import all relevant packages throughout this walkthrough

In [3]:
# Modules for fine-tuning
from unsloth import FastLanguageModel
import torch # Import PyTorch
from trl import SFTTrainer # Trainer for supervised fine-tuning (SFT)
from unsloth import is_bfloat16_supported # Checks if the hardware supports bfloat16 precision
# Hugging Face modules
from huggingface_hub import login # Lets you login to API
from transformers import TrainingArguments # Defines training hyperparameters
from datasets import load_dataset # Lets you load fine-tuning datasets
# Import weights and biases
import wandb
# Import kaggle secrets
from kaggle_secrets import UserSecretsClient

import pandas as pd
import re
import string
from sklearn.metrics import precision_score, recall_score
import random
from typing import List, Dict, Tuple

# Import necessary packages
import gradio as gr
import torch
import re
import os
from pathlib import Path
import warnings

# Document processing and retrieval
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma

# Embedding generation using HuggingFace embeddings instead of Ollama
from langchain_community.embeddings import HuggingFaceEmbeddings

# Import kaggle secrets for token management
from kaggle_secrets import UserSecretsClient

from transformers import pipeline

## Import all relevant packages
import torch
import re
import os
import requests
from pathlib import Path
from bs4 import BeautifulSoup
import youtube_dl
import whisper

# Hugging Face modules
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Unsloth and model imports
from unsloth import FastLanguageModel
from transformers import pipeline

# Document processing and retrieval
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.schema import Document

# Alternative embedding using transformers directly
from transformers import AutoTokenizer, AutoModel
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-06-11 23:25:54.132604: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749684354.566710      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749684354.679940      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


In [5]:
# warnings.filterwarnings("ignore", message="Unexpected keyword arguments")

## Create API keys and login to Hugging Face and Weights and Biases

In [6]:
user_secrets = UserSecretsClient()
hugging_face_token = user_secrets.get_secret("HF_TOKEN_DEEPSEEK")
wnb_token = user_secrets.get_secret("wnb_token")

login(hugging_face_token)

wandb.login(key=wnb_token)
run = wandb.init(
    project='Fine-tune-DeepSeek-R1-Distill-Llama-8B on LawBot', 
    job_type="training", 
    anonymous="allow"
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: naufalkr394 (naufalkr394-institut-teknologi-sepuluh-nopember) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Loading Model and the Tokenizer

In [7]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

# Load model adapter menggunakan Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="luckysantoso/adapter_deepseek_lawbot",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    token=hugging_face_token,
)

FastLanguageModel.for_inference(model)

/usr/local/lib/python3.11/dist-packages/peft/config.py:162: UserWarning: Unexpected keyword arguments ['alpha_pattern', 'bias', 'corda_config', 'eva_config', 'exclude_modules', 'fan_in_fan_out', 'init_lora_weights', 'layer_replication', 'layers_pattern', 'layers_to_transform', 'loftq_config', 'lora_alpha', 'lora_bias', 'lora_dropout', 'megatron_config', 'megatron_core', 'modules_to_save', 'r', 'rank_pattern', 'target_modules', 'trainable_token_indices', 'use_dora', 'use_rslora'] for class PeftConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


==((====))==  Unsloth 2025.6.2: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/53.0k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/peft/config.py:162: UserWarning: Unexpected keyword arguments ['corda_config', 'trainable_token_indices'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


adapter_model.safetensors:   0%|          | 0.00/37.8M [00:00<?, ?B/s]

Unsloth 2025.6.2 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (

## Testing Model on a law use-case before RAG

In [8]:
def test_model_basic():
    """Test basic model functionality"""
    print("\n🧪 Testing basic model functionality...")
    
    prompt_style = """
Di bawah ini adalah instruksi yang menjelaskan tugas, dipasangkan dengan input yang memberikan konteks lebih lanjut.
Tuliskan respons yang menyelesaikan permintaan dengan tepat.
Sebelum menjawab, pikirkan dengan cermat pertanyaan tersebut dan buatlah rangkaian pemikiran langkah demi langkah untuk memastikan respons yang logis dan akurat.

### Instruksi:
Anda adalah seorang ahli hukum dengan pengetahuan tingkat lanjut dalam penalaran hukum, analisis kasus, dan penyusunan dokumen hukum. 
Jawablah pertanyaan hukum berikut ini dengan tepat, berdasarkan peraturan perundang-undangan yang berlaku dan preseden hukum yang relevan.

### Pertanyaan:
{}

### Jawaban:
<think>
"""

    # question = """Apa arti dari "berada di bawah Presiden" dalam konteks TNI?"""
    question = """Bagaimana dampak revisi UU TNI 2025 terhadap peran TNI?"""

    inputs = tokenizer([prompt_style.format(question)], return_tensors="pt").to("cuda")

    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=800,
        use_cache=True,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id else tokenizer.pad_token_id
    )

    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    print("Basic test response:")
    print(response.split("### Jawaban:")[-1] if "### Jawaban:" in response else response)

In [9]:
test_model_basic()


🧪 Testing basic model functionality...
Basic test response:

<think>
Sure, I'll help you with the question about the impact of the revision of the TNI Law 2025 on the role of the TNI.
</think>

Revisi Undang-Undang TNI 2025 memberikan beberapa perubahan yang penting terhadap peran TNI. Berikut adalah ringkasan utamanya:

1. **Pengertian dan Tujuan Baru**: Undang-Undang yang diperbarui menyebutkan tujuan untuk mempertahankan keamanan negara dan memberikan batas kerja yang jelas bagi TNI.

2. **Peran Operatif yang Terpisah**: TNI kini memiliki fungsi khusus dalam mengimplementasikan keputusan superior dan bertindak sebagai agen utama keamanan negara.

3. **Kewenangan yang Terpisah**: TNI memiliki kewenangan khusus dalam bidang keamanan, termasuk operasi militer dan intelijen, sementara fungsi lainnya seperti penjagaan keamanan rumah dan sosial dipotongkan.

4. **Penghapusan Fungsi Lain**: Perubahan ini menghapuskan fungsi TNI sebagai penyelamat bencana dan penjagaan keamanan rumah, sehi

# **RAG PEFT Implementation**

In [10]:
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F
from rouge_score import rouge_scorer
from bert_score import score
import json
from tqdm import tqdm
import re
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi
import numpy as np
from sentence_transformers import CrossEncoder

In [11]:
# Embedding model for Indonesian text
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
# embedding_model_name = "firqaaa/indo-sentence-bert-base"

embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

# Cross-encoder for re-ranking
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

In [12]:
# Text preprocessing with Indonesian-specific handling
def preprocess_text(text):
    """Enhanced text preprocessing for Indonesian legal documents"""
    if not isinstance(text, str):
        return ""
    
    # Remove extra whitespace and normalize
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    # Keep Indonesian legal terms and abbreviations
    text = re.sub(r'[^\w\s\-.,;:!?()\[\]/]', '', text)
    
    # Normalize common legal abbreviations
    legal_replacements = {
        'Undang-Undang': 'UU',
        'undang-undang': 'UU',
        'Tentara Nasional Indonesia': 'TNI',
        'Operasi Militer Selain Perang': 'OMSP',
        'Angkatan Bersenjata Republik Indonesia': 'ABRI'
    }
    
    for old, new in legal_replacements.items():
        text = text.replace(old, new)
    
    return text

In [13]:
# import re
# import pandas as pd
# from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
# from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
# from langchain.schema import Document
# from langchain.document_loaders import PyMuPDFLoader

# # Initialize Sastrawi components
# stemmer_factory = StemmerFactory()
# stemmer = stemmer_factory.create_stemmer()

# stopword_factory = StopWordRemoverFactory()
# stopword_remover = stopword_factory.create_stop_word_remover()

# def preprocess_youtube_transcript_sastrawi(text):
#     """Preprocess YouTube transcript using Sastrawi for legal document context"""
#     if not text or pd.isna(text):
#         return ""
    
#     text = str(text)
    
#     # Step 1: Remove timestamps and sound effects
#     text = re.sub(r'\b\d{1,2}:\d{2}(?::\d{2})?\b', '', text)
    
#     sound_effects = [
#         r'\[musik\]', r'\[Musik\]', r'\[MUSIK\]',
#         r'\[tepuk tangan\]', r'\[Tepuk tangan\]', r'\[TEPUK TANGAN\]',
#         r'\[applause\]', r'\[Applause\]', r'\[APPLAUSE\]',
#         r'\[laughter\]', r'\[Laughter\]', r'\[LAUGHTER\]',
#         r'\[tawa\]', r'\[Tawa\]', r'\[TAWA\]',
#         r'\[suara\s+\w+\]', r'\[Suara\s+\w+\]'
#     ]
    
#     for pattern in sound_effects:
#         text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    
#     # Step 2: Remove unclear/foreign elements
#     unclear_patterns = [
#         r'\b[a-zA-Z]{1,2}\b',  # Very short English words
#         r'\b\w*[xX]{2,}\w*\b',  # Words with multiple x's
#         r'\b\w*[zZ]{2,}\w*\b',  # Words with multiple z's
#     ]
    
#     for pattern in unclear_patterns:
#         text = re.sub(pattern, '', text)
    
#     # Step 3: Normalize informal Indonesian (before Sastrawi)
#     informal_mapping = {
#         'gue': 'saya', 'gw': 'saya', 'ane': 'saya', 'gua': 'saya',
#         'lu': 'kamu', 'lo': 'kamu', 'elu': 'kamu', 'loe': 'kamu',
#         'udah': 'sudah', 'udh': 'sudah', 'dah': 'sudah',
#         'emang': 'memang', 'emg': 'memang', 'mang': 'memang',
#         'gimana': 'bagaimana', 'gmn': 'bagaimana',
#         'kenapa': 'mengapa', 'knp': 'mengapa', 'napa': 'mengapa',
#         'bgt': 'banget', 'banget': 'sangat', 'bener': 'benar',
#         'yg': 'yang', 'dgn': 'dengan', 'sm': 'sama',
#         'krn': 'karena', 'krna': 'karena', 'soalnya': 'karena',
#         'tp': 'tetapi', 'tapi': 'tetapi',
#         'klo': 'kalau', 'kalo': 'kalau', 'kl': 'kalau',
#         'ga': 'tidak', 'gak': 'tidak', 'nggak': 'tidak', 'engga': 'tidak',
#         'org': 'orang', 'orng': 'orang', 'orgnya': 'orangnya',
#         'tau': 'tahu', 'gatau': 'tidak tahu',
#         'aja': 'saja', 'doang': 'saja', 'aj': 'saja',
#         'abis': 'habis', 'abs': 'habis', 'stelah': 'setelah',
#         'blm': 'belum', 'blom': 'belum', 'msh': 'masih',
#         'jg': 'juga', 'jga': 'juga', 'jd': 'jadi'
#     }
    
#     # Apply informal word mapping
#     words = text.split()
#     normalized_words = []
#     for word in words:
#         word_lower = word.lower()
#         if word_lower in informal_mapping:
#             normalized_words.append(informal_mapping[word_lower])
#         else:
#             normalized_words.append(word)
#     text = ' '.join(normalized_words)
    
#     # Step 4: Clean whitespace
#     text = re.sub(r'\s+', ' ', text).strip()
    
#     # Step 5: Preserve important legal terms before Sastrawi processing
#     legal_terms_preserve = {
#         'undang-undang': 'UNDANGUNDANG_PRESERVE',
#         'tni': 'TNI_PRESERVE',
#         'tentara nasional indonesia': 'TENTARANASIONALINDONESIA_PRESERVE',
#         'pasal': 'PASAL_PRESERVE',
#         'ayat': 'AYAT_PRESERVE',
#         'peraturan': 'PERATURAN_PRESERVE',
#         'pemerintah': 'PEMERINTAH_PRESERVE',
#         'militer': 'MILITER_PRESERVE',
#         'pertahanan': 'PERTAHANAN_PRESERVE',
#         'keamanan': 'KEAMANAN_PRESERVE'
#     }
    
#     # Preserve legal terms
#     for term, placeholder in legal_terms_preserve.items():
#         text = re.sub(r'\b' + re.escape(term) + r'\b', placeholder, text, flags=re.IGNORECASE)
    
#     # Step 6: Apply Sastrawi processing (conservative approach)
#     # Remove common stopwords but preserve important words
#     custom_stopwords = ['di', 'ke', 'dari', 'pada', 'untuk', 'dengan', 'oleh', 'dalam', 'sebagai', 'adalah', 'akan', 'telah', 'sudah', 'dapat', 'bisa', 'juga', 'atau', 'dan', 'ini', 'itu', 'yang', 'ada', 'tidak', 'ya', 'iya']
    
#     words = text.split()
#     filtered_words = []
#     for word in words:
#         if word.lower() not in custom_stopwords or len(word) <= 2:
#             filtered_words.append(word)
#         else:
#             filtered_words.append(word)
#     text = ' '.join(filtered_words)
    
#     # Apply stemming (conservative)
#     text = stemmer.stem(text)
    
#     # Step 7: Restore preserved legal terms
#     for term, placeholder in legal_terms_preserve.items():
#         text = text.replace(placeholder, term)
    
#     # Step 8: Basic punctuation and capitalization
#     # Add periods after common sentence endings
#     text = re.sub(r'\b(jadi|maka|sehingga|oleh karena itu|dengan demikian)\s+', r'\1. ', text, flags=re.IGNORECASE)
#     text = re.sub(r'\b(ya|iya|benar|betul|memang)\s+', r'\1. ', text, flags=re.IGNORECASE)
    
#     # Clean up punctuation
#     text = re.sub(r'\s*,\s*', ', ', text)
#     text = re.sub(r'\s*\.\s*', '. ', text)
#     text = text.strip()
    
#     # Capitalize sentences
#     sentences = text.split('. ')
#     capitalized_sentences = []
#     for sentence in sentences:
#         sentence = sentence.strip()
#         if sentence:
#             sentence = sentence[0].upper() + sentence[1:] if len(sentence) > 1 else sentence.upper()
#             capitalized_sentences.append(sentence)
#     text = '. '.join(capitalized_sentences)
    
#     # Step 9: Final quality check
#     words = text.split()
#     if len(words) < 3 or len(text) < 20:
#         return ""
    
#     return text

# def calculate_transcript_quality_score_sastrawi(text):
#     """Calculate quality score for Sastrawi-processed YouTube transcript"""
#     if not text or len(text.strip()) < 10:
#         return 0
    
#     score = 0
    
#     # Length score (adjusted for stemmed text)
#     length = len(text)
#     if length > 150:  # Lower threshold due to stemming
#         score += 20
#     elif length > 80:
#         score += 15
#     elif length > 40:
#         score += 10
    
#     # Word count score
#     words = text.split()
#     word_count = len(words)
#     if word_count > 25:  # Adjusted for stemmed content
#         score += 20
#     elif word_count > 12:
#         score += 15
#     elif word_count > 5:
#         score += 10
    
#     # Legal/formal terms (bonus for relevant content)
#     legal_terms = ['undang', 'pasal', 'ayat', 'tni', 'tentara', 'nasional', 
#                    'indonesia', 'hukum', 'peraturan', 'kebijakan', 'pemerintah',
#                    'militer', 'pertahanan', 'keamanan', 'negara', 'republik']
#     legal_term_count = sum(1 for term in legal_terms if term.lower() in text.lower())
#     score += min(legal_term_count * 6, 25)  # Higher weight for legal terms
    
#     # Sentence structure
#     sentences = text.split('.')
#     complete_sentences = [s for s in sentences if len(s.strip().split()) > 2]
#     if len(complete_sentences) > 0:
#         score += min(len(complete_sentences) * 4, 20)
    
#     # Bonus for formal language indicators
#     formal_indicators = ['berdasarkan', 'sesuai', 'menurut', 'sebagaimana', 'dimaksud']
#     formal_count = sum(1 for word in formal_indicators if word in text.lower())
#     score += min(formal_count * 3, 15)
    
#     return max(0, min(score, 100))

# def load_youtube_transcripts_sastrawi(csv_path, quality_threshold=35):
#     """Load and process YouTube transcript data with Sastrawi integration"""
#     documents = []
    
#     try:
#         print(f"Loading YouTube transcripts with Sastrawi from: {csv_path}")
#         df = pd.read_csv(csv_path)
        
#         total_videos = len(df)
#         processed_count = 0
#         quality_filtered = 0
        
#         for _, row in df.iterrows():
#             title = str(row.get('Title', ''))
#             script = str(row.get('Script', ''))
#             date = str(row.get('Date', ''))
#             url = str(row.get('URL', ''))
            
#             # Preprocess with Sastrawi
#             processed_script = preprocess_youtube_transcript_sastrawi(script)
            
#             if not processed_script:
#                 continue
            
#             # Calculate quality score
#             quality_score = calculate_transcript_quality_score_sastrawi(processed_script)
            
#             # Filter by quality threshold
#             if quality_score < quality_threshold:
#                 quality_filtered += 1
#                 continue
            
#             # Enhanced content combination
#             full_content = f"Judul Video: {title}\n\nTranskrip: {processed_script}"
#             full_content = preprocess_text(full_content)  # Apply existing preprocessing
            
#             if len(full_content) > 100:
#                 documents.append(Document(
#                     page_content=full_content,
#                     metadata={
#                         "source": url,
#                         "title": title,
#                         "date": date,
#                         "type": "youtube_transcript",
#                         "quality_score": quality_score,
#                         "content_length": len(full_content),
#                         "original_length": len(script),
#                         "preprocessing": "sastrawi_enhanced"
#                     }
#                 ))
#                 processed_count += 1
        
#         print(f"YouTube Transcripts (Sastrawi) Summary:")
#         print(f"  Total videos: {total_videos}")
#         print(f"  Processed successfully: {processed_count}")
#         print(f"  Filtered by quality: {quality_filtered}")
#         print(f"  Final documents: {len(documents)}")
        
#     except Exception as e:
#         print(f"Error loading YouTube transcripts: {e}")
    
#     return documents

In [14]:
# Document loading 
def load_pdf_documents(pdf_paths):
    """Load and process PDF documents with metadata"""
    documents = []
    
    for pdf_path in pdf_paths:
        try:
            print(f"Loading PDF: {pdf_path}")
            loader = PyMuPDFLoader(pdf_path)
            pages = loader.load()
            
            # Determine document type from filename
            doc_type = "unknown"
            if "naskah-akademik" in pdf_path.lower():
                doc_type = "academic_draft"
            elif "petisi" in pdf_path.lower():
                doc_type = "petition"
            elif "permohonan" in pdf_path.lower():
                doc_type = "legal_request"
            elif "wiraedsus" in pdf_path.lower():
                doc_type = "analysis_report"
            
            for page in pages:
                content = preprocess_text(page.page_content)
                if len(content) > 100:  # Increased minimum length
                    documents.append(Document(
                        page_content=content,
                        metadata={
                            "source": pdf_path,
                            "page": page.metadata.get("page", 0),
                            "type": "pdf",
                            "doc_type": doc_type,
                            "content_length": len(content)
                        }
                    ))
            print(f"Loaded {len([d for d in documents if d.metadata['source'] == pdf_path])} pages from {pdf_path}")
            
        except Exception as e:
            print(f"Error loading {pdf_path}: {e}")
    
    return documents

def load_web_articles(csv_path):
    """Load and process web scraping data with metadata"""
    documents = []
    
    try:
        print(f"Loading web articles from: {csv_path}")
        df = pd.read_csv(csv_path)
        
        for _, row in df.iterrows():
            title = str(row.get('Title', ''))
            content = str(row.get('Content', ''))
            date = str(row.get('Date', ''))
            url = str(row.get('URL', ''))
            
            # Enhanced content combination
            full_content = f"Judul: {title}\n\nIsi: {content}"
            full_content = preprocess_text(full_content)
            
            if len(full_content) > 100:  # Increased minimum length
                documents.append(Document(
                    page_content=full_content,
                    metadata={
                        "source": url,
                        "title": title,
                        "date": date,
                        "type": "web_article",
                        "content_length": len(full_content)
                    }
                ))
        
        print(f"Loaded {len(documents)} web articles")
        
    except Exception as e:
        print(f"Error loading web articles: {e}")
    
    return documents

# Load all documents
PDF_SOURCES = [
    "/kaggle/input/rag-pdf/Permohonan_4288_8190_Permohonan_redact.pdf",
    "/kaggle/input/rag-pdf/Petisi-Revisi-UU-TNI-.pdf", 
    "/kaggle/input/rag-pdf/naskah-akademik.pdf",
    "/kaggle/input/rag-pdf/wiraedsus2019-web.pdf"
]
WEB_CSV_PATH = "/kaggle/input/rag-web-scrape-v3/detik_articles.csv"
YOUTUBE_CSV_PATH = "/kaggle/input/rag-vid-v3/YT-scrape (1).csv"

pdf_documents = load_pdf_documents(PDF_SOURCES)
web_documents = load_web_articles(WEB_CSV_PATH)
# youtube_documents = load_youtube_transcripts_sastrawi(YOUTUBE_CSV_PATH, quality_threshold=35)

# all_documents = pdf_documents + web_documents + youtube_documents
all_documents = pdf_documents + web_documents

Loading PDF: /kaggle/input/rag-pdf/Permohonan_4288_8190_Permohonan_redact.pdf
Loaded 24 pages from /kaggle/input/rag-pdf/Permohonan_4288_8190_Permohonan_redact.pdf
Loading PDF: /kaggle/input/rag-pdf/Petisi-Revisi-UU-TNI-.pdf
Loaded 11 pages from /kaggle/input/rag-pdf/Petisi-Revisi-UU-TNI-.pdf
Loading PDF: /kaggle/input/rag-pdf/naskah-akademik.pdf
Loaded 28 pages from /kaggle/input/rag-pdf/naskah-akademik.pdf
Loading PDF: /kaggle/input/rag-pdf/wiraedsus2019-web.pdf
Loaded 56 pages from /kaggle/input/rag-pdf/wiraedsus2019-web.pdf
Loading web articles from: /kaggle/input/rag-web-scrape-v3/detik_articles.csv
Loaded 239 web articles


In [15]:
# Text splitting
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,  # Reduced for better coherence
    chunk_overlap=150,  # Better overlap ratio
    length_function=len,
    separators=["\n\n", "\n", ". ", "! ", "? ", "; ", ", ", " "],  # More specific separators
    is_separator_regex=False
)

chunks = text_splitter.split_documents(all_documents)

# Filter out low-quality chunks
quality_chunks = []
for chunk in chunks:
    content = chunk.page_content.strip()
    # Filter criteria
    if (len(content) > 50 and 
        len(content.split()) > 10 and  # At least 10 words
        not content.lower().startswith(('halaman', 'page', 'gambar', 'tabel'))):
        quality_chunks.append(chunk)

print(f"Total chunks created: {len(quality_chunks)}")


Total chunks created: 1516


In [16]:
# Ganti nama collection untuk model baru
vectorstore = Chroma.from_documents(
    documents=quality_chunks,
    embedding=embeddings,
    persist_directory="./enhanced_chroma_db_v3",  # Nama folder berbeda
    # collection_name="indo_bert_collection",  # Nama collection berbeda
    collection_metadata={"hnsw:space": "cosine"}
)

In [17]:
# Initialize BM25 for hybrid retrieval
print("🔧 Initializing BM25 for hybrid retrieval...")
document_texts = [chunk.page_content for chunk in quality_chunks]
tokenized_docs = [doc.lower().split() for doc in document_texts]
bm25 = BM25Okapi(tokenized_docs)

🔧 Initializing BM25 for hybrid retrieval...


In [18]:
# Hybrid retrieval function with semantic + BM25
def retrieve_documents_hybrid(query, k=8, alpha=0.7):
    """
    Hybrid retrieval combining semantic similarity and BM25
    alpha: weight for semantic similarity (1-alpha for BM25)
    """
    # 1. Semantic similarity search
    semantic_results = vectorstore.similarity_search_with_score(query, k=k*2)
    
    # 2. BM25 search
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    
    # Get top BM25 results
    bm25_indices = np.argsort(bm25_scores)[::-1][:k*2]
    
    # 3. Combine and normalize scores
    combined_results = {}
    
    # Add semantic results
    for doc, score in semantic_results:
        doc_index = None
        for i, chunk in enumerate(quality_chunks):
            if chunk.page_content == doc.page_content:
                doc_index = i
                break
        
        if doc_index is not None:
            # Normalize semantic score (higher is better, so invert distance)
            normalized_semantic = 1 / (1 + score)
            combined_results[doc_index] = {
                'doc': doc,
                'semantic_score': normalized_semantic,
                'bm25_score': 0
            }
    
    # Add BM25 results
    max_bm25_score = max(bm25_scores) if max(bm25_scores) > 0 else 1
    for idx in bm25_indices:
        normalized_bm25 = bm25_scores[idx] / max_bm25_score
        
        if idx in combined_results:
            combined_results[idx]['bm25_score'] = normalized_bm25
        else:
            combined_results[idx] = {
                'doc': quality_chunks[idx],
                'semantic_score': 0,
                'bm25_score': normalized_bm25
            }
    
    # 4. Calculate combined scores
    for idx in combined_results:
        semantic_score = combined_results[idx]['semantic_score']
        bm25_score = combined_results[idx]['bm25_score']
        combined_results[idx]['combined_score'] = alpha * semantic_score + (1 - alpha) * bm25_score
    
    # 5. Sort by combined score and return top k
    sorted_results = sorted(combined_results.items(), 
                          key=lambda x: x[1]['combined_score'], 
                          reverse=True)
    
    return [result[1]['doc'] for result in sorted_results[:k]]

# Re-ranking mechanism using cross-encoder
def rerank_documents(query, documents, top_k=6):
    """
    Re-rank retrieved documents using cross-encoder
    """
    if len(documents) <= top_k:
        return documents
    
    # Prepare query-document pairs for cross-encoder
    pairs = [[query, doc.page_content] for doc in documents]
    
    # Get relevance scores from cross-encoder
    scores = cross_encoder.predict(pairs)
    
    # Sort documents by relevance scores
    doc_score_pairs = list(zip(documents, scores))
    doc_score_pairs.sort(key=lambda x: x[1], reverse=True)
    
    # Return top-k re-ranked documents
    return [doc for doc, score in doc_score_pairs[:top_k]]

# Retrieval with hybrid + re-ranking
def retrieve_documents(query, k=8):
    """
    Enhanced retrieval with hybrid approach and re-ranking
    """
    # Step 1: Hybrid retrieval (semantic + BM25)
    hybrid_results = retrieve_documents_hybrid(query, k=k*2)
    
    # Step 2: Re-ranking with cross-encoder
    reranked_results = rerank_documents(query, hybrid_results, top_k=k)
    
    # Step 3: Ensure diversity (same as before)
    results = []
    seen_sources = set()
    
    for doc in reranked_results:
        source_key = f"{doc.metadata.get('source', '')}_{doc.metadata.get('page', 0)}"
        if source_key not in seen_sources:
            results.append(doc)
            seen_sources.add(source_key)
        elif len(results) < k//2:  # Allow some duplicates if needed
            results.append(doc)
    
    return results[:k]

In [19]:
# Response generation 
def generate_rag_response(query, max_new_tokens=1000):
    """Enhanced RAG response generation with hybrid retrieval"""
    # Retrieve relevant documents using hybrid approach
    retrieved_docs = retrieve_documents(query, k=6)
    
    # Better context organization
    context_parts = []
    for i, doc in enumerate(retrieved_docs, 1):
        doc_type = doc.metadata.get('doc_type', doc.metadata.get('type', 'unknown'))
        source_info = f"Sumber {i} ({doc_type})"
        context_parts.append(f"{source_info}:\n{doc.page_content}")
    
    context = "\n\n".join(context_parts)
    
    prompt_style = f"""Di bawah ini adalah instruksi yang menjelaskan tugas, dipasangkan dengan konteks yang memberikan informasi relevan.
Tuliskan respons yang menyelesaikan permintaan dengan tepat berdasarkan konteks yang diberikan.

### Instruksi:
Anda adalah seorang ahli hukum Indonesia dengan spesialisasi dalam hukum militer dan TNI. Berikan jawaban yang komprehensif, akurat, dan berbasis bukti dari konteks yang tersedia. Jika informasi tidak lengkap dalam konteks, sebutkan hal tersebut.

### Konteks Relevan:
{context[:3000]}

### Pertanyaan:
{query}

### Jawaban:
Berdasarkan konteks yang tersedia, saya dapat menjelaskan sebagai berikut:

"""

    inputs = tokenizer([prompt_style], return_tensors="pt").to("cuda")
    
    # Better generation parameters
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=0.3,  # Lower temperature for more focused answers
        top_p=0.9,  # Nucleus sampling
        do_sample=True,
        repetition_penalty=1.1,  # Prevent repetition
        pad_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id else tokenizer.pad_token_id
    )
    
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    answer = response.split("### Jawaban:")[-1].strip() if "### Jawaban:" in response else response
    
    # Clean up the response
    answer = answer.replace("Berdasarkan konteks yang tersedia, saya dapat menjelaskan sebagai berikut:", "").strip()
    
    return answer, retrieved_docs

In [20]:
# Evaluation metrics
def calculate_precision_recall(retrieved_docs, expected_contexts, k_values=[3, 5, 8]):
    """Enhanced precision and recall calculation"""
    results = {}
    
    for k in k_values:
        retrieved_k = retrieved_docs[:k]
        retrieved_texts = [doc.page_content.lower() for doc in retrieved_k]
        retrieved_full = " ".join(retrieved_texts)
        
        # Better matching using multiple criteria
        found = 0
        partial_matches = 0
        
        for expected in expected_contexts:
            expected_lower = expected.lower()
            expected_words = set(expected_lower.split())
            
            # Exact phrase match
            if expected_lower in retrieved_full:
                found += 1
            # Partial match (at least 50% of words)
            else:
                retrieved_words = set(retrieved_full.split())
                overlap = len(expected_words.intersection(retrieved_words))
                if overlap >= len(expected_words) * 0.5:
                    partial_matches += 1
        
        # Include partial matches with weight
        effective_found = found + (partial_matches * 0.5)
        
        precision_k = effective_found / k if k > 0 else 0
        recall_k = effective_found / len(expected_contexts) if len(expected_contexts) > 0 else 0
        
        results[f"precision@{k}"] = precision_k
        results[f"recall@{k}"] = recall_k
        results[f"exact_matches@{k}"] = found
        results[f"partial_matches@{k}"] = partial_matches
    
    return results

In [22]:
ground_truth = [
    {
        "query": "Apa fungsi utama TNI menurut UU TNI?",
        "expected_contexts": [
            "fungsi TNI sebagai alat pertahanan negara",
            "TNI bertugas menjaga kedaulatan",
            "pertahanan dan keamanan negara", 
            "melindungi keutuhan wilayah NKRI",
            "mempertahankan kemerdekaan Indonesia",
            "menghadapi ancaman militer",
            "menjaga integritas teritorial"
        ]
    },
    {
        "query": "Apa perbedaan UU TNI lama dan revisi 2025?",
        "expected_contexts": [
            "revisi UU TNI 2025",
            "perubahan UU TNI",
            "UU TNI yang baru",
            "perbedaan UU lama dan baru",
            "pembaruan undang-undang TNI",
            "batas usia pensiun perwira",
            "perluasan peran OMSP",
            "jabatan sipil untuk militer aktif"
        ]
    },
    {
        "query": "Bagaimana TNI berperan dalam operasi militer selain perang?",
        "expected_contexts": [
            "operasi militer selain perang",
            "OMSP TNI",
            "penanggulangan bencana alam",
            "bantuan kemanusiaan",
            "operasi non-perang",
            "misi perdamaian internasional",
            "kontra-terorisme",
            "pengamanan perbatasan",
            "search and rescue SAR"
        ]
    },
    {
        "query": "Apa kritik terhadap revisi UU TNI 2025?",
        "expected_contexts": [
            "kritik terhadap UU TNI",
            "penolakan revisi UU TNI",
            "kontroversi UU TNI 2025",
            "keberatan masyarakat sipil",
            "pro kontra revisi",
            "kekhawatiran dwifungsi TNI",
            "pelanggaran supremasi sipil",
            "transparansi pembahasan",
            "demokratisasi Indonesia"
        ]
    },
    {
        "query": "Bagaimana hubungan TNI dengan pemerintah sipil?",
        "expected_contexts": [
            "TNI di bawah Presiden",
            "supremasi sipil atas militer",
            "hubungan sipil militer",
            "kontrol sipil terhadap TNI",
            "pemerintahan demokratis",
            "akuntabilitas TNI",
            "profesionalisme militer",
            "netralitas TNI dalam politik"
        ]
    },
    {
        "query": "Apa itu dwifungsi TNI dan mengapa menjadi perdebatan?",
        "expected_contexts": [
            "dwifungsi TNI ABRI",
            "peran sosial politik militer",
            "dwifungsi era Orde Baru",
            "reformasi sektor militer",
            "perdebatan dwifungsi",
            "supremasi sipil vs militer",
            "demokrasi dan militer",
            "pembatasan peran politik TNI",
            "reformasi 1998"
        ]
    },
    {
        "query": "Bagaimana proses pengangkatan perwira tinggi TNI?",
        "expected_contexts": [
            "pengangkatan perwira tinggi TNI",
            "mutasi dan promosi TNI",
            "jabatan militer strategis",
            "promosi perwira senior",
            "karier militer TNI",
            "persetujuan DPR",
            "evaluasi kinerja perwira",
            "sistem kepangkatan militer",
            "fit and proper test"
        ]
    }
]

In [25]:
reference_answers = [
    "TNI berfungsi sebagai alat pertahanan negara yang bertugas menjaga kedaulatan dan keutuhan wilayah NKRI dari ancaman militer. TNI berperan mempertahankan kemerdekaan, melindungi segenap bangsa Indonesia, dan menjaga integritas teritorial dari gangguan terhadap keutuhan bangsa dan negara sesuai dengan UU TNI.",
    "Revisi UU TNI 2025 mengatur perubahan signifikan meliputi: perpanjangan batas usia pensiun perwira tinggi, perluasan peran dalam OMSP, pengaturan jabatan sipil yang dapat diisi militer aktif, penyesuaian ancaman keamanan modern, dan penguatan koordinasi dengan lembaga keamanan lainnya.",
    "TNI berperan dalam OMSP meliputi: penanggulangan bencana alam, bantuan kemanusiaan internasional, misi penjaga perdamaian PBB, operasi kontra-terorisme, pengamanan wilayah perbatasan, operasi SAR, dan bantuan kepada masyarakat dalam situasi darurat sesuai kebijakan pemerintah.",
    "Kritik terhadap revisi UU TNI 2025 meliputi: kekhawatiran kembalinya dwifungsi TNI, kurangnya transparansi pembahasan, potensi pelanggaran supremasi sipil, perluasan kewenangan militer yang berlebihan, dan kekhawatiran terhadap kemunduran demokratisasi serta netralitas TNI dalam politik.",
    "TNI berada di bawah komando Presiden sebagai Panglima Tertinggi dan tunduk pada supremasi sipil dalam sistem demokrasi Indonesia. Hubungan ini diatur dengan prinsip akuntabilitas, transparansi, profesionalisme militer, dan netralitas politik untuk menjaga keseimbangan kekuasaan demokratis.",
    "Dwifungsi TNI adalah konsep peran ganda militer dalam pertahanan-keamanan dan sosial-politik yang diterapkan era Orde Baru. Konsep ini kontroversial karena melanggar prinsip supremasi sipil dan demokrasi, sehingga dibatasi pasca reformasi 1998 untuk menjaga netralitas militer dan penguatan institusi sipil.",
    "Pengangkatan perwira tinggi TNI dilakukan Presiden dengan pertimbangan DPR melalui fit and proper test. Proses melibatkan evaluasi kinerja, kompetensi, integritas, dan loyalitas sesuai sistem karier militer, dengan mempertimbangkan profesionalisme dan netralitas politik untuk menjaga kredibilitas institusi."
]

In [26]:
def calculate_precision_recall(retrieved_docs, expected_contexts, k_values=[3, 5, 8]):
    """Fixed precision and recall calculation"""
    results = {}
    
    for k in k_values:
        retrieved_k = retrieved_docs[:k]
        retrieved_texts = [doc.page_content.lower() for doc in retrieved_k]
        
        # Track which documents are relevant (per document scoring)
        relevant_docs = 0
        exact_matches = 0
        partial_matches = 0
        
        # Score each document individually
        for doc in retrieved_k:
            doc_content = doc.page_content.lower()
            doc_relevant = False
            doc_exact = False
            
            for expected in expected_contexts:
                expected_lower = expected.lower()
                
                # Exact phrase match
                if expected_lower in doc_content:
                    doc_exact = True
                    doc_relevant = True
                    break
                # Partial match (at least 50% of words)
                else:
                    expected_words = set(expected_lower.split())
                    doc_words = set(doc_content.split())
                    overlap = len(expected_words.intersection(doc_words))
                    if overlap >= len(expected_words) * 0.5:
                        doc_relevant = True
            
            if doc_relevant:
                relevant_docs += 1
                if doc_exact:
                    exact_matches += 1
                else:
                    partial_matches += 1
        
        # Calculate metrics (now bounded by k)
        precision_k = relevant_docs / k if k > 0 else 0
        
        # For recall, count unique expected contexts found
        found_contexts = 0
        retrieved_full = " ".join(retrieved_texts)
        
        for expected in expected_contexts:
            expected_lower = expected.lower()
            if expected_lower in retrieved_full:
                found_contexts += 1
            else:
                expected_words = set(expected_lower.split())
                retrieved_words = set(retrieved_full.split())
                overlap = len(expected_words.intersection(retrieved_words))
                if overlap >= len(expected_words) * 0.5:
                    found_contexts += 0.5
        
        recall_k = found_contexts / len(expected_contexts) if len(expected_contexts) > 0 else 0
        
        results[f"precision@{k}"] = precision_k
        results[f"recall@{k}"] = recall_k
        results[f"exact_matches@{k}"] = exact_matches  # ✅ FIXED: gunakan exact_matches
        results[f"partial_matches@{k}"] = partial_matches
    
    return results

## With Video Dataset

In [27]:
all_scores = []
generated_answers_arr = []

for i, item in enumerate(ground_truth):
    query = item["query"]
    expected_contexts = item["expected_contexts"]
    
    print(f"\nEvaluating query {i+1}: {query}")
    
    # Generate enhanced RAG response
    answer, retrieved_docs = generate_rag_response(query)
    generated_answers_arr.append(answer)
    
    # Calculate enhanced metrics
    scores = calculate_precision_recall(retrieved_docs, expected_contexts)
    all_scores.append(scores)
    
    # Display all k values dengan format yang rapi
    print(f"Answer preview: {answer[:100]}...")
    for k in [3, 5, 8]:
        precision = scores[f'precision@{k}']
        recall = scores[f'recall@{k}']
        exact = scores[f'exact_matches@{k}']
        partial = scores[f'partial_matches@{k}']
        
        print(f"  k={k}: Precision={precision:.3f}, Recall={recall:.3f} | Exact={exact}, Partial={partial}")


Evaluating query 1: Apa fungsi utama TNI menurut UU TNI?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Answer preview: TNImberikan informasi yang komprehensif, akurat, dan berdasarkan bukti dari konteks yang diberikan.
...
  k=3: Precision=0.667, Recall=0.214 | Exact=0, Partial=2
  k=5: Precision=0.400, Recall=0.214 | Exact=0, Partial=2
  k=8: Precision=0.250, Recall=0.214 | Exact=0, Partial=2

Evaluating query 2: Apa perbedaan UU TNI lama dan revisi 2025?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Answer preview: Pertama, UU TNI lama adalah UU Nomor 34 Tahun 2004 yang memberikan batasan dan regulasi bagi TNI dal...
  k=3: Precision=1.000, Recall=0.375 | Exact=0, Partial=3
  k=5: Precision=1.000, Recall=0.375 | Exact=0, Partial=5
  k=8: Precision=0.625, Recall=0.375 | Exact=0, Partial=5

Evaluating query 3: Bagaimana TNI berperan dalam operasi militer selain perang?


...

Answer preview: 1. **Perkembangan Peran TNI Sipil**  
   Menurut sumber kedua, TNI aktif mulai menduduki jabatan sip...
  k=3: Precision=1.000, Recall=0.438 | Exact=0, Partial=3
  k=5: Precision=0.600, Recall=0.438 | Exact=0, Partial=3
  k=8: Precision=0.375, Recall=0.438 | Exact=0, Partial=3

Evaluating query 6: Apa itu dwifungsi TNI dan mengapa menjadi perdebatan?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Answer preview: Dwifungsionalitas TNI adalah kondisi dimana unit TNI dapat melakukan kedua tujuan utamanya yaitu per...
  k=3: Precision=1.000, Recall=0.500 | Exact=0, Partial=3
  k=5: Precision=0.600, Recall=0.500 | Exact=0, Partial=3
  k=8: Precision=0.375, Recall=0.500 | Exact=0, Partial=3

Evaluating query 7: Bagaimana proses pengangkatan perwira tinggi TNI?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Answer preview: Pengangkatan perwira tinggi TNI melibatkan beberapa langkah penting yang ditetapkan sesuai dengan re...
  k=3: Precision=1.000, Recall=0.278 | Exact=0, Partial=3
  k=5: Precision=0.800, Recall=0.333 | Exact=0, Partial=4
  k=8: Precision=0.500, Recall=0.333 | Exact=0, Partial=4


In [28]:
avg_scores = {}
for k in [3, 5, 8]:
    avg_scores[f"avg_precision@{k}"] = np.mean([scores[f"precision@{k}"] for scores in all_scores])
    avg_scores[f"avg_recall@{k}"] = np.mean([scores[f"recall@{k}"] for scores in all_scores])
    avg_scores[f"avg_exact_matches@{k}"] = np.mean([scores[f"exact_matches@{k}"] for scores in all_scores])

In [29]:
def evaluate_with_bert_score(generated_answers, reference_answers):
    """
    BERT Score evaluation with robust fallback models
    """
    # Daftar model BERT yang akan dicoba secara berurutan
    models_to_try = [
        # "bert-base-multilingual-cased",  # Paling stabil
        # "distilbert-base-multilingual-cased",  # Lebih cepat
        "xlm-roberta-base",  # Bagus untuk multilingual
    ]
    
    for model_name in models_to_try:
        try:
            P, R, F1 = score(
                generated_answers,
                reference_answers,
                model_type=model_name,
                lang="id",
                verbose=False,
                device="cuda" if torch.cuda.is_available() else "cpu",
                rescale_with_baseline=True  
            )
            
            print(f"BERT Model: {model_name}")
            return {
                "bert_precision": P.mean().item(),
                "bert_recall": R.mean().item(),
                "bert_f1": F1.mean().item(),
                "bert_precision_std": P.std().item(),
                "bert_recall_std": R.std().item(),
                "bert_f1_std": F1.std().item(),
                "model_used": model_name
            }
            
        except Exception as e:
            print(f"{model_name} failed to use: {e}")
            continue
    
    # Jika semua model gagal, gunakan evaluasi sederhana
    print("Semua model BERT gagal, menggunakan evaluasi fallback")
    return {
        "bert_precision": 0.0,
        "bert_recall": 0.0,
        "bert_f1": 0.0,
        "bert_precision_std": 0.0,
        "bert_recall_std": 0.0,
        "bert_f1_std": 0.0,
        "model_used": "none"
    }


# BERT evaluation
bert_scores = evaluate_with_bert_score(generated_answers_arr, reference_answers)

print("\n" + "="*70)
print("RAG EVALUATION RESULTS")
print("="*70)

print("\nPrecision@k and Recall@k Results:")
for k in [3, 5, 8]:
    print(f"Average Precision@{k}: {avg_scores[f'avg_precision@{k}']:.4f}")
    print(f"Average Recall@{k}: {avg_scores[f'avg_recall@{k}']:.4f}")
    print(f"Average Exact Matches@{k}: {avg_scores[f'avg_exact_matches@{k}']:.2f}")
    
print("\nBERT Score Results:")
print(f"BERT Precision: {bert_scores['bert_precision']:.4f}")
print(f"BERT Recall: {bert_scores['bert_recall']:.4f}")
print(f"BERT F1: {bert_scores['bert_f1']:.4f}")


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

BERT Model: xlm-roberta-base

RAG EVALUATION RESULTS

Precision@k and Recall@k Results:
Average Precision@3: 0.9524
Average Recall@3: 0.3292
Average Exact Matches@3: 0.43
Average Precision@5: 0.7714
Average Recall@5: 0.3530
Average Exact Matches@5: 0.43
Average Precision@8: 0.4821
Average Recall@8: 0.3530
Average Exact Matches@8: 0.43

BERT Score Results:
BERT Precision: 0.7980
BERT Recall: 0.8704
BERT F1: 0.8326


In [108]:
# Test RAG system
print("\n" + "="*70)
print("TESTING RAG SYSTEM")
print("="*70)

test_queries = [
    "Apa saja tugas pokok TNI menurut UU TNI?",
    "Bagaimana dampak revisi UU TNI 2025 terhadap peran TNI?",
    "Mengapa terjadi penolakan terhadap revisi UU TNI?",
    "Bagaimana aksi demo UU TNI di Surabaya berjalan?",
    "Bagaimana mekanisme pertanggungjawaban TNI kepada rakyat?"
]

for query in test_queries:
    print(f"\nQuery: {query}")
    answer, retrieved_docs = generate_rag_response(query)
    print(f"RAG Answer: {answer}")
    print(f"Retrieved {len(retrieved_docs)} relevant documents")
    
    # Show source diversity
    source_types = [doc.metadata.get('doc_type', doc.metadata.get('type', 'unknown')) for doc in retrieved_docs]
    print(f"Source types: {', '.join(set(source_types))}")
    print("-" * 50)


TESTING ENHANCED RAG SYSTEM

Query: Apa saja tugas pokok TNI menurut UU TNI?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

RAG Answer: **Tugas Pokok TNI Menurut UU TNI:**

1. **Pertahanan Negara:** TNI bertindak sebagai garda terakhir dalam mengamankan negara dari ancaman internal maupun external.
2. **Kebajikan Masyarakat:** Melakukan operasi darurat untuk membantu masyarakat saat musim buruh atau bencana.
3. **Dukung Kebharian:** Memberikan bantuan teknis dan logistik kepada instansi lain seperti kepolisian, kejaksaan, dan lainnya.
4. **Reformasi Birokrasi:** Memfasilitasi percepatan proses kejahatan dengan menggunakan sarana prajuritnya.
5. **Kebanggaan Negara:** Menjaga citra positif dan profesionalisme TNI di mata publik.

Namun, informasi tentang batas-batas pelibatan TNI di bidang nonmiliter dan hubungan dengan kejaksaan serta modernisasi alutsista belum sepenuhnya tercakup dalam konteks yang diberikan. Untuk informasi lebih lengkap, mungkin perlu merujuk pada UU TNI terbaru atau dokumen resmi lainnya.
</think>

**Tugas Pokok TNI Menurut UU TNI:**

1. **Pertahanan Negara:** TNI bertindak sebagai gar

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

RAG Answer: Pertama, penting untuk memahami maksud revisi UU TNI 2025. Dari sumber 1 dan 3, diketahui bahwa revisi ini ditujukan untuk memperbaiki sistem hukum militer Indonesia dan meningkatkan efektivitas TNI. Namun, proses penyusunannya tidak transparan dan tidak mencerminkan nilai demokratis, sehingga menyebabkan kritik dari masyarakat sipil (sumber 1).

Selanjutnya, perubahan yang direncanakan termasuk perluasan penempatan TNI di kementerian dan lembaga lainnya (sumber 2). Ini berarti TNI tidak lagi hanya bertindak sebagai pasukan militer tetapi memiliki peran yang lebih luas dalam pemerintahan.

Namun, ada juga masalah yang terkait dengan jabatan sipil yang bisa diduduki oleh prajurit aktif dan penambahan usia pensiun (sumber 2). Ini mungkin menyebabkan kesulitan dalam pengembangan karier prajurit dan mempengaruhi kualitas tenaga manusia TNI.

Selain itu, UU 3 Tahun 2025 tentang Perubahan atas UU 34 Tahun 2004 tidak tercantum dalam agenda rapat paripurna DPR RI (sumber 3 dan 4). 

...

RAG Answer: Penolakan terhadap revisi UU TNI terjadi karena beberapa alasan utama:

1. **Kurangnya Transparansi dalam Proses Legislasi**:
   - Menurut Dewa Palguna, proses penyusunan UU TNI tidak hanya tertutup, tetapi juga tidak sesuai dengan prosedur legislasi yang ideal. Revisi UU TNI tidak dimasukkan dalam Prolegnas dan tidak dapat diakses publik. Ini menyebabkan kurangnya akuntabilitas dan transparansi dalam proses legislasi.
   
2. **Kebimbangan tentang Restorasi Supremasi Sipil**:
   - Rachmat Hidayat, anggota Komisi I DPR, menyatakan bahwa pengesahan revisi UU TNI tidak akan menghidupkan kembali dwifungsi ABRI. Namun, politikus PDIP tersebut juga menambahkan bahwa revisi tersebut justru memperkuat supremasi sipil dan menjadikan TNI lebih baik. Ini menunjukkan kebimbangan bahwa revisi tersebut mungkin mengganggu supremasi sipil yang telah terbangun setelah Orde Baru.

3. **Kefasihuan Aspek Hukum dan Politik**:
   - Mahasiswa yang turut dalam aksi tersebut menyoroi bahwa revisi U

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

RAG Answer: Pertama, aksi dimulai dengan para peserta demo menyatakan tujuan utama mereka yaitu menolak UU TNI karena merasa fungsi militer TNI sudah terlalu dominan dalam ruang siber masyarakat. Mereka juga menuntut agar TNI tidak terlibat dalam urusan sipil dan untuk mengembalikan TNI ke barak. Aksi ini digelar di depan gedung Negara Grahadi, Surabaya, dan diketuai oleh seorang yang dikenali sebagai Jaya.

Selanjutnya, peserta demo menyampaikan delapan poin tuntutan yang ditandai dengan berbagai poin penting seperti tolak fungsi TNI di ranah sipil, tolak UU TNI yang sedang direvisi, dan lainnya. Aksi ini didukung oleh warga lokal dan organisasi mahasiswa, sehingga jumlah peserta cukup banyak.

Namun, selama aksi, ada beberapa insiden kecil yang terjadi, seperti bentrokan dengan aparat kepolisian yang menyebabkan beberapa orang diamukan dan lainnya mendapat luka-luka. Ini menunjukkan bahwa meskipun aksi dimaksudkan sebagai suatu yang damai, tetapi tidak semua aspeknya berhasil berjala

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

RAG Answer: **Mechanism of Accountability of TNI to the People**

1. **Pengertian Dasar**
   - TNI adalah institusi militer negara yang bertanggung jawab langsung kepada Presiden sebagai Komandan tertinggi.
   - Tanggung jawab TNI juga ditujukan kepada rakyat Indonesia sebagai pelindidik keamanan nasional dan martabat bangsa.

2. **Rencana Kerja dan Tujuan**
   - TNI memiliki tujuan untuk meningkatkan kualitas pelayanan keamanan publik.
   - Melalui program-program pengembangan kapasitas, TNI berusaha meningkatkan efektivitas operasinya.

3. **Kebijakan Publik dan Transparansi**
   - Informasi tentang operasi dan kegiatan TNI disampaikan melalui kanal resmi seperti laporan kepresiden dan dokumen parlementer.
   - Keputusan-keputusan penting TNI biasanya diproses melalui prosedur demokrasif untuk memastikan ketelusuran.

4. **Mechanism Pengawasan**
   - Dewan Pengawasan TNI (Dewan Kehakiman Militer) bertindak sebagai pengawasinternal atas kegiatan TNI.
   - Badan Parlemen Indonesia juga

In [109]:
# Test RAG system
print("\n" + "="*70)
print("TESTING ENHANCED RAG SYSTEM")
print("="*70)

test_queries = [
    "Apakah RUU TNI 2025 sudah disahkan menjadi UU? Jika sudah, disahkan sebagai UU nomor berapa?"
]

for query in test_queries:
    print(f"\nQuery: {query}")
    answer, retrieved_docs = generate_rag_response(query)
    print(f"RAG Answer: {answer}")
    print(f"Retrieved {len(retrieved_docs)} relevant documents")
    
    # Show source diversity
    source_types = [doc.metadata.get('doc_type', doc.metadata.get('type', 'unknown')) for doc in retrieved_docs]
    print(f"Source types: {', '.join(set(source_types))}")
    print("-" * 50)


TESTING ENHANCED RAG SYSTEM

Query: Apakah RUU TNI 2025 sudah disahkan menjadi UU? Jika sudah, disahkan sebagai UU nomor berapa?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

RAG Answer: Pertama, RUU Revisi UU TNI yang ditetapkan sebagai UU Nomor 3 Tahun 2025 Tentang Perubahan atas UU Nomor 34 Tahun 2004 Tentang TNI (TNI) telah disahkan oleh DPR RI melalui rapat paripurna pada 18 Februari 2025. Ini dilakukan sebagai langkah awal untuk mengubah dan memperbarui UU yang relevan dengan kondisi saat ini.

Namun demikian, perlu juga diperhatikan bahwa RUU Revisi UU TNI tidak tercantum dalam Prolegnas 2025 dan tidak termasuk dalam 18 RUU prioritas yang ditetapkan dalam Rencana Pembangunan Jangka Menengah Nasional (RPJMN) 2025-2029. Ini menunjukkan bahwa proses legislasi untuk RUU ini terjadi secara terpisah dari proses legislasi umum tahun tersebut.

Selain itu, meskipun RUU Revisi UU TNI sudah disahkan, tetap ada kontroversi dan kritik dari beberapa pihak terkait dengan proses penetapannya. Pakar hukum menyebutkan bahwa proses ini terlalu cepat dan mengabaikan partisipasi yang bermakna dari masyarakat dan instansi terkait.

Untuk informasi lebih lengkap, mungkin 

In [131]:
# def create_gradio_interface():
#     """Create a Gradio web interface for the chatbot"""
    
#     def chat_interface(message, history):
#         try:
#             response = generate_response(message, use_rag=True)
#             history.append((message, response))
#             return history, ""
#         except Exception as e:
#             error_msg = f"Error: {str(e)}"
#             history.append((message, error_msg))
#             return history, ""
    
#     with gr.Blocks(title="RAG LawBot TNI") as interface:
#         gr.Markdown("# RAG LawBot TNI")
#         gr.Markdown("Chatbot hukum TNI dengan fine-tuned model dan RAG")
        
#         chatbot = gr.Chatbot(height=400)
#         msg = gr.Textbox(placeholder="Tanyakan sesuatu tentang UU TNI...")
#         clear = gr.Button("Clear")
        
#         msg.submit(chat_interface, [msg, chatbot], [chatbot, msg])
#         clear.click(lambda: [], None, chatbot, queue=False)
    
#     return interface

In [132]:
# interface = create_simple_gradio_interface()
# interface.launch(share=True)